In [20]:
# =====================================================
# Razorpay Buildathon - Payment Failure Prediction
# Steps 1 & 2: Load Data + Feature Engineering
# =====================================================

import pandas as pd
import os

In [21]:
# Ensure the data folder exists before reading/writing anything into it
os.makedirs("data", exist_ok=True)

In [23]:
# -----------------------------
# STEP 1: Load and inspect data
# -----------------------------
df = pd.read_csv("transactions_dataset.csv")

print("Dataset shape:", df.shape)

print("\nColumn info:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTarget class balance:")
print(df["Status"].value_counts())

print("\nFirst 5 rows:")
print(df.head())


Dataset shape: (1000, 8)

Column info:
Transaction ID      object
Timestamp           object
Sender Name         object
Sender UPI ID       object
Receiver Name       object
Receiver UPI ID     object
Amount (INR)       float64
Status              object
dtype: object

Missing values per column:
Transaction ID     0
Timestamp          0
Sender Name        0
Sender UPI ID      0
Receiver Name      0
Receiver UPI ID    0
Amount (INR)       0
Status             0
dtype: int64

Target class balance:
Status
SUCCESS    502
FAILED     498
Name: count, dtype: int64

First 5 rows:
                         Transaction ID            Timestamp      Sender Name  \
0  4d3db980-46cd-4158-a812-dcb77055d0d2  2024-06-22 04:06:38        Tiya Mall   
1  099ee548-2fc1-4811-bf92-559c467ca792  2024-06-19 06:04:49  Mohanlal Bakshi   
2  d4c05732-6b1b-4bab-90b9-efe09d252b99  2024-06-04 04:56:09      Kismat Bora   
3  e8df92ee-8b04-4133-af5a-5f412180c8ab  2024-06-09 09:56:07    Ayesha Korpal   
4  e7d675d3-04f1

In [24]:
# -----------------------------
# STEP 2: Feature Engineering
# -----------------------------

# --- Extract time-based features from Timestamp ---
# Why: Raw timestamps are useless to a model directly, but the HOUR and
# DAY OF WEEK inside them often carry real patterns (e.g., failures spiking
# during peak evening hours, or on weekends due to higher load).
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["hour"] = df["Timestamp"].dt.hour
df["day_of_week"] = df["Timestamp"].dt.dayofweek  # 0=Monday, 6=Sunday

In [25]:
# --- Extract bank/provider from UPI IDs ---
# Why: The part after '@' in a UPI ID (e.g., okaxis, okhdfcbank, oksbi)
# identifies the bank/PSP handling that side of the transaction.
# Different banks can have different failure rates due to infra differences.
df["sender_bank"] = df["Sender UPI ID"].str.split("@").str[1]
df["receiver_bank"] = df["Receiver UPI ID"].str.split("@").str[1]

In [26]:
# --- Feature: same bank transaction or not ---
# Why: Transactions within the same bank often succeed more reliably than
# cross-bank transactions, which depend on inter-bank settlement systems.
df["same_bank"] = (df["sender_bank"] == df["receiver_bank"]).astype(int)

In [27]:
# --- Drop columns we don't need for modeling ---
# Why: Transaction ID, Sender Name, Receiver Name, and raw UPI IDs are
# unique identifiers — they don't generalize, they'd just let the model
# "memorize" instead of learning real patterns. Original Timestamp is
# dropped since we've already extracted the useful parts from it.
df_model = df.drop(columns=[
    "Transaction ID", "Timestamp", "Sender Name", "Receiver Name",
    "Sender UPI ID", "Receiver UPI ID"
])


In [32]:
print("\nShape after feature engineering:", df_model.shape)



Shape after feature engineering: (1000, 7)


In [33]:
print("\nColumns now:", list(df_model.columns))



Columns now: ['Amount (INR)', 'Status', 'hour', 'day_of_week', 'sender_bank', 'receiver_bank', 'same_bank']


In [34]:
print("\nSample rows:")
print(df_model.head())


Sample rows:
   Amount (INR)   Status  hour  day_of_week sender_bank receiver_bank  \
0       3907.34   FAILED     4            5      okaxis         okybl   
1       8404.55  SUCCESS     6            2      okaxis        okaxis   
2        941.88  SUCCESS     4            1       okybl       okicici   
3       8926.00  SUCCESS     9            6  okhdfcbank        okaxis   
4       2800.55  SUCCESS     8            1       okybl         okybl   

   same_bank  
0          0  
1          1  
2          0  
3          0  
4          1  


In [35]:
# Save this intermediate cleaned dataset for the next step
df_model.to_csv("transactions_features.csv", index=False)
print("\nSaved engineered dataset to transactions_features.csv")


Saved engineered dataset to transactions_features.csv


In [36]:
# step3_encode_and_split.py

import pandas as pd
from sklearn.model_selection import train_test_split


In [37]:
# -----------------------------
# STEP 3: Encoding + Train/Test Split
# -----------------------------

df_model = pd.read_csv("transactions_features.csv")

In [38]:
# --- Encode target column ---
# Why: Models need numbers, not text. We map FAILED=1, SUCCESS=0 so that
# "1" represents the thing we actually care about predicting (failure).
df_model["Status"] = df_model["Status"].map({"SUCCESS": 0, "FAILED": 1})

In [39]:
# --- One-hot encode categorical bank columns ---
# Why: sender_bank/receiver_bank are text categories (okaxis, oksbi, etc).
# One-hot encoding turns each unique bank into its own 0/1 column,
# so the model can use them without assuming any false numeric order.
df_encoded = pd.get_dummies(df_model, columns=["sender_bank", "receiver_bank"], drop_first=True)

In [40]:
print("Shape after encoding:", df_encoded.shape)
print("\nColumns now:", list(df_encoded.columns))

Shape after encoding: (1000, 13)

Columns now: ['Amount (INR)', 'Status', 'hour', 'day_of_week', 'same_bank', 'sender_bank_okhdfcbank', 'sender_bank_okicici', 'sender_bank_oksbi', 'sender_bank_okybl', 'receiver_bank_okhdfcbank', 'receiver_bank_okicici', 'receiver_bank_oksbi', 'receiver_bank_okybl']


In [41]:
# --- Separate features (X) and target (y) ---
X = df_encoded.drop(columns=["Status"])
y = df_encoded["Status"]

In [42]:
# --- Train/test split ---
# Why: We train the model on one portion of data (80%) and test it on
# unseen data (20%) to check if it actually generalizes, not just
# memorizes the training data. random_state ensures reproducibility.
# stratify=y keeps the SUCCESS/FAILED ratio same in both train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [43]:
print("\nTraining set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


Training set shape: (800, 12)
Test set shape: (200, 12)


In [44]:
# Save these splits so the next script (model training) can load them directly
X_train.to_csv("data/X_train.csv", index=False)
X_test.to_csv("data/X_test.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)
y_test.to_csv("data/y_test.csv", index=False)

print("\nSaved train/test splits to data/ folder")


Saved train/test splits to data/ folder


In [45]:
# step4_model_building.py

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

In [46]:
# -----------------------------
# STEP 4: Model Building
# -----------------------------

# Load the train/test splits saved in Step 3
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").values.ravel()
y_test = pd.read_csv("data/y_test.csv").values.ravel()

In [47]:
# --- Model 1: Logistic Regression (baseline) ---
# Why: Logistic Regression is simple, fast, and interpretable. It's our
# baseline — if a more complex model can't beat this, it's not worth using.
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)
print("Logistic Regression trained.")

Logistic Regression trained.


In [48]:
# --- Model 2: Random Forest ---
# Why: Random Forest can capture non-linear patterns and feature
# interactions (like hour + bank combinations) that Logistic Regression
# might miss. It usually performs better on tabular data like this.
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
print("Random Forest trained.")

Random Forest trained.


In [49]:
# --- Save both trained models ---
# Why: So we can reload them later for evaluation (Step 5) and for the
# demo app (Step 8), without retraining every time.
joblib.dump(log_model, "data/logistic_model.pkl")
joblib.dump(rf_model, "data/random_forest_model.pkl")


['data/random_forest_model.pkl']

In [50]:
print("\nBoth models saved to data/ folder.")


Both models saved to data/ folder.


In [51]:
# step5_evaluation.py

import pandas as pd
import joblib
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

In [52]:
# -----------------------------
# STEP 5: Evaluation
# -----------------------------

# Load test data
X_test = pd.read_csv("data/X_test.csv")
y_test = pd.read_csv("data/y_test.csv").values.ravel()


In [53]:
# Load trained models
log_model = joblib.load("data/logistic_model.pkl")
rf_model = joblib.load("data/random_forest_model.pkl")

def evaluate_model(model, name):
    preds = model.predict(X_test)

    print(f"\n===== {name} =====")
    print("Accuracy: ", accuracy_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds))
    print("Recall:   ", recall_score(y_test, preds))
    print("F1 Score: ", f1_score(y_test, preds))

    print("\nConfusion Matrix:")
    # Rows = actual, Columns = predicted
    # [[TN, FP],
    #  [FN, TP]]
    print(confusion_matrix(y_test, preds))
    print("\nFull classification report:")
    print(classification_report(y_test, preds, target_names=["SUCCESS", "FAILED"]))

    return preds

In [54]:
# Evaluate both models
log_preds = evaluate_model(log_model, "Logistic Regression")
rf_preds = evaluate_model(rf_model, "Random Forest")


===== Logistic Regression =====
Accuracy:  0.53
Precision: 0.5288461538461539
Recall:    0.55
F1 Score:  0.5392156862745098

Confusion Matrix:
[[51 49]
 [45 55]]

Full classification report:
              precision    recall  f1-score   support

     SUCCESS       0.53      0.51      0.52       100
      FAILED       0.53      0.55      0.54       100

    accuracy                           0.53       200
   macro avg       0.53      0.53      0.53       200
weighted avg       0.53      0.53      0.53       200


===== Random Forest =====
Accuracy:  0.465
Precision: 0.4666666666666667
Recall:    0.49
F1 Score:  0.47804878048780486

Confusion Matrix:
[[44 56]
 [51 49]]

Full classification report:
              precision    recall  f1-score   support

     SUCCESS       0.46      0.44      0.45       100
      FAILED       0.47      0.49      0.48       100

    accuracy                           0.47       200
   macro avg       0.46      0.46      0.46       200
weighted avg       0.

In [55]:
# -----------------------------
# Why precision/recall matter here:
# - Precision: of all transactions we predicted as FAILED, how many
#   actually failed? Low precision = too many false alarms.
# - Recall: of all transactions that actually failed, how many did we
#   correctly catch? Low recall = missing real failures, which is risky
#   for a merchant relying on this to act proactively.
# - F1: balances both, useful for comparing models overall.
# -----------------------------

In [56]:
# step6_edge_case_testing.py

import pandas as pd
import joblib

In [57]:
# -----------------------------
# STEP 6: Edge Case Testing
# -----------------------------

rf_model = joblib.load("data/random_forest_model.pkl")
X_train = pd.read_csv("data/X_train.csv")  # to check expected columns

print("Expected feature columns:", list(X_train.columns))

Expected feature columns: ['Amount (INR)', 'hour', 'day_of_week', 'same_bank', 'sender_bank_okhdfcbank', 'sender_bank_okicici', 'sender_bank_oksbi', 'sender_bank_okybl', 'receiver_bank_okhdfcbank', 'receiver_bank_okicici', 'receiver_bank_oksbi', 'receiver_bank_okybl']


In [59]:
# --- Edge Case 1: A brand-new bank not seen during training ---
# Why: One-hot encoding only knows banks present in training data.
# If a transaction comes from a bank the model never saw, all its
# bank-related columns will just be 0 — the model has no real signal
# for that case. This is a genuine limitation worth documenting.
edge_case_1 = X_train.iloc[0:1].copy()
edge_case_1.loc[:, :] = 0  # zero out everything to simulate "unknown bank"
edge_case_1["Amount (INR)"] = 5000
edge_case_1["hour"] = 14
edge_case_1["day_of_week"] = 2
edge_case_1["same_bank"] = 0

pred_1 = rf_model.predict(edge_case_1)
prob_1 = rf_model.predict_proba(edge_case_1)
print("\nEdge Case 1 (unknown bank, mid-range amount):")
print("Prediction:", "FAILED" if pred_1[0] == 1 else "SUCCESS")
print("Confidence:", prob_1[0])


Edge Case 1 (unknown bank, mid-range amount):
Prediction: FAILED
Confidence: [0.485 0.515]


C:\Users\User Name\AppData\Local\Temp\ipykernel_10688\285297075.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  edge_case_1.loc[:, :] = 0  # zero out everything to simulate "unknown bank"
C:\Users\User Name\AppData\Local\Temp\ipykernel_10688\285297075.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  edge_case_1.loc[:, :] = 0  # zero out everything to simulate "unknown bank"
C:\Users\User Name\AppData\Local\Temp\ipykernel_10688\285297075.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  edge_case_1

In [60]:
# --- Edge Case 2: Extremely high transaction amount ---
# Why: Check if the model behaves sensibly for amounts far outside the
# range it was trained on (training max was ~9993 INR).
edge_case_2 = X_train.iloc[0:1].copy()
edge_case_2.loc[:, :] = 0
edge_case_2["Amount (INR)"] = 100000  # far beyond training range
edge_case_2["hour"] = 3
edge_case_2["day_of_week"] = 5
edge_case_2["same_bank"] = 1

pred_2 = rf_model.predict(edge_case_2)
prob_2 = rf_model.predict_proba(edge_case_2)
print("\nEdge Case 2 (very high amount, same bank):")
print("Prediction:", "FAILED" if pred_2[0] == 1 else "SUCCESS")
print("Confidence:", prob_2[0])


Edge Case 2 (very high amount, same bank):
Prediction: SUCCESS
Confidence: [0.645 0.355]


C:\Users\User Name\AppData\Local\Temp\ipykernel_10688\1273705755.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  edge_case_2.loc[:, :] = 0
C:\Users\User Name\AppData\Local\Temp\ipykernel_10688\1273705755.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  edge_case_2.loc[:, :] = 0
C:\Users\User Name\AppData\Local\Temp\ipykernel_10688\1273705755.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  edge_case_2.loc[:, :] = 0
C:\Users\User Name\AppData\Local\Temp\ipykernel_10688\1273705755.py:5: FutureWarni

In [61]:
# -----------------------------
# WHAT TO DOCUMENT (for your pitch video / README):
# - The model has no reliable signal for banks not seen in training —
#   this is a real limitation of one-hot encoding on a small dataset.
# - Fix/mitigation: in a production system, you'd use a fallback
#   "unknown_bank" category during encoding, or retrain periodically
#   as new banks appear.
# - This is your "one thing that broke and how I'd fix it" story.
# -----------------------------

In [1]:
import os
print(os.getcwd())

C:\Users\User Name
